In [1]:
from database.adatabase import ADatabase
import pandas as pd
from processor.processor import Processor as p
from tqdm import tqdm

In [2]:
market = ADatabase("market")

In [9]:
market.connect()
sp500 = market.retrieve("sp500")
spy = market.retrieve("spy")
market.disconnect()

In [10]:
spy = p.column_date_processing(spy)

In [13]:
analysis = []
market.connect()
analysis = []
for ticker in tqdm(sp500["ticker"]):
    try:
        ticker_prices = p.column_date_processing(market.query("prices",{"ticker":ticker}))
        ticker_prices = p.merge(ticker_prices,spy,"date")
        corr = ticker_prices["adjclose"].corr(ticker_prices["value"])
        corr = 1 if corr > 0.5 else 0 if corr < 0.5 and corr > -0.5 else -1
        analysis.append({"ticker":ticker,"correlation":corr})
    except Exception as e:
        print(str(e))
        continue
market.disconnect()

 13%|█████████████████                                                                                                                   | 65/503 [00:05<00:37, 11.75it/s]

'date'


 16%|████████████████████▉                                                                                                               | 80/503 [00:07<00:33, 12.44it/s]

'date'


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 503/503 [00:46<00:00, 10.77it/s]

'date'


In [15]:
market.connect()
market.drop("spy_correlations")
market.store("spy_correlations",pd.DataFrame(analysis))
market.disconnect()
